In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")

eigenval_path = 'data/bbj.pca_base.eigenval'
sscore_path = 'data/cteph_agp3k_v5_wgs_merged.sample_qc.variant_qc.bbjproj.sscore'

In [2]:
# STEP0, Data Loading (BBJ + OUR cohort) from scripts/data_loading.py
import importlib
import scripts.data_loading as data_loading_mod

importlib.reload(data_loading_mod)
from scripts.data_loading import (
    DataLoadingConfig,
    load_step0_bbj_and_our,
 )

config_step0 = DataLoadingConfig(
    chunksize=50000,
    bbj_prefix="bbj_",
    verbose=True,
    phenotype_column="PHENO1",
    case_value=2,
    control_value=1,
 )

# Single-entry STEP0 loader: eigenval + BBJ + OUR (+ case/control IID lists)
eigenval, bbj_samples, our_samples, our_case_iids, our_ctrl_iids = load_step0_bbj_and_our(
    eigenval_path=eigenval_path,
    sscore_path=sscore_path,
    config=config_step0,
 )


                        STEP0: DATA LOADING (BBJ + OUR)                         

[CONFIGURATION]
--------------------------------------------------------------------------------
  Eigenvalue path      : data/bbj.pca_base.eigenval
  Score file path      : data/cteph_agp3k_v5_wgs_merged.sample_qc.variant_qc.bbjproj.sscore
  Chunk size           : 50,000
  BBJ prefix           : bbj_
  Phenotype column     : PHENO1
  Case / Control value : 2 / 1

[RESULTS]
--------------------------------------------------------------------------------
  Eigenvalues          : 20 PCs × 4 metrics
  PC1 variance explained: 39.11%
  PC1-2 cumulative var.: 46.56%
  BBJ samples          : 183,013 × 22 cols (49.92 MB)
  OUR samples          : 3,532 × 22 cols (0.93 MB)
  OUR cases / ctrls    : 406 / 3,126



In [3]:
# STEP1, BBJ HDBSCAN Denoising (focused + memory-efficient)
import importlib
import scripts.hdbscan_filtering as hdbscan_filtering_mod
import gc

importlib.reload(hdbscan_filtering_mod)
from scripts.hdbscan_filtering import HDBSCANConfig, run_hdbscan_denoise_bbj

# HDBSCAN is used to remove sparse outliers/noise in PCA space
# and retain stable population structure for downstream modeling.
config_step1 = HDBSCANConfig(
    n_pcs_hdbscan=2,
    use_zscale_hdbscan=True,
    min_cluster_size=50,
    min_samples=6,
    cluster_selection_epsilon=0.005,
    cluster_selection_method="eom",
    metric="euclidean",
    alpha=0.8,
    allow_single_cluster=True,
    leaf_size=40,
    algorithm="best",
    approx_min_span_tree=True,
    gen_min_span_tree=False,
    output_dir="results/01_hdbscan_filtering",
    save_plot=True,
    save_tables=True,
    save_full_table=False,
    verbose=True,
)

bbj_hdbscan = run_hdbscan_denoise_bbj(
    bbj_samples=bbj_samples,
    eigenval=eigenval,
    config=config_step1,
)

# Main downstream input: only non-noise BBJ samples.
bbj_samples_filtered = bbj_hdbscan.bbj_samples_filtered.drop(columns=["HDBSCAN_Label"], errors="ignore")
hdb_summary = bbj_hdbscan.summary

# Release large intermediates after successful run.
del bbj_hdbscan
del bbj_samples
_ = gc.collect()


                         STEP1: HDBSCAN DENOISING (BBJ)                         

[CONFIGURATION]
--------------------------------------------------------------------------------
  n_pcs_hdbscan         : 2
  min_cluster_size      : 50
  min_samples           : 6
  cluster_epsilon       : 0.005
  cluster_method        : eom
  metric                : euclidean
  alpha                 : 0.8
  allow_single_cluster  : True
  leaf_size             : 40
  algorithm             : best
  approx_min_span_tree  : True
  gen_min_span_tree     : False
  use_zscale            : True
  save_plot             : True
  save_full_table       : False
  output_dir            : results/01_hdbscan_filtering

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 183,013
  output_rows           : 181,639
  noise_rows            : 1,374
  noise_ratio           : 0.75%
  clusters_found        : 7


In [4]:
# STEP2, BBJ GMM Clustering (fixed PCs + min BIC)
import importlib
import scripts.gmm_search_audit as gmm_search_audit_mod
import scripts.gmm_clustering as gmm_clustering_mod
import gc

# Reload dependency first, then the main clustering module.
importlib.reload(gmm_search_audit_mod)
importlib.reload(gmm_clustering_mod)
from scripts.gmm_clustering import GMMConfig, run_gmm_fixed_pcs

# Fix the number of PCs to 2, and search for optimal K by BIC.
config_step2 = GMMConfig(
    fixed_n_pcs=2,
    k_min=2,
    k_max=30,
    use_zscale=False,
    covariance_type="full",
    n_init=3,
    init_params="kmeans",
    reg_covar=1e-6,
    max_iter=200,
    random_state=42,
    search_max_samples=200000,
    search_workers=6,
    require_non_empty_clusters=True,
    output_dir="results/02_gmm_clustering",
    save_plot=True,
    save_tables=True,
    verbose=True,
)

gmm_result = run_gmm_fixed_pcs(
    bbj_samples_filtered=bbj_samples_filtered,
    eigenval=eigenval,
    config=config_step2,
)

# Main downstream outputs.
bbj_samples_gmm = gmm_result.bbj_samples_with_cluster
gmm_bic_table = gmm_result.bic_table
gmm_cluster_summary = gmm_result.cluster_summary
gmm_summary = gmm_result.summary

# Keep fitted model for downstream merging/diagnostics.
gmm_model = gmm_result.model

# Release large intermediates after successful run.
del gmm_result
_ = gc.collect()


                   STEP2: GMM CLUSTERING (FIXED PCs, MIN-BIC)                   

[CONFIGURATION]
--------------------------------------------------------------------------------
  fixed_n_pcs           : 2
  k_range               : 2..30
  covariance_type       : full
  n_init                : 3
  init_params           : kmeans
  reg_covar             : 1e-06
  max_iter              : 200
  random_state          : 42
  search_rows           : 181,639
  full_rows             : 181,639
  search_workers        : 6
  require_non_empty     : True
  use_zscale            : False
  output_dir            : results/02_gmm_clustering

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,639
  best_k                : 26
  best_bic              : -2,680,131.67
  non_empty_models      : 29
  clusters_found        : 26


In [5]:
# STEP3, Merge nearby GMM components by Mahalanobis distance + hierarchical clustering
import importlib
import scripts.gmm_component_merging as gmm_component_merging_mod

importlib.reload(gmm_component_merging_mod)
from scripts.gmm_component_merging import GMMComponentMergingConfig, run_gmm_component_merging

config_step3 = GMMComponentMergingConfig(
    merge_threshold=5.0,
    linkage_method="average",
    output_dir="results/03_gmm_component_merging",
    save_plot=True,
    save_tables=True,
    # Panel D: stable and interpretable legend range across runs
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    # Compress differences near 1.0 (high-confidence end)
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_result = run_gmm_component_merging(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step3,
)

# Common downstream variables (kept for notebook convenience).
merge_map = merge_result.merge_map
label_map = merge_result.label_map


             STEP3: GMM COMPONENT MERGING (MAHALANOBIS + H-CLUSTER)             

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 5.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  output_dir            : results/03_gmm_component_merging

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,639
  original_components   : 26
  merged_components     : 7


In [22]:
# STEP3-TMP, Re-run component merging for inspection only (merge_threshold=2.5)
import importlib
from pathlib import Path
import scripts.gmm_component_merging as gmm_component_merging_mod

importlib.reload(gmm_component_merging_mod)
from scripts.gmm_component_merging import GMMComponentMergingConfig, run_gmm_component_merging

tmp_output_dir = "results/04_tmp_gmm_component_merging_mt2p5"
Path(tmp_output_dir).mkdir(parents=True, exist_ok=True)

config_step3_tmp = GMMComponentMergingConfig(
    merge_threshold=2.5,
    linkage_method="average",
    output_dir=tmp_output_dir,
    save_plot=True,
    save_tables=True,
    # Keep the same confidence color scaling for side-by-side comparison
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_result_tmp = run_gmm_component_merging(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step3_tmp,
)

# TEMP outputs for visual inspection only; do not use as downstream inputs.
merge_map_tmp = merge_result_tmp.merge_map
label_map_tmp = merge_result_tmp.label_map


             STEP3: GMM COMPONENT MERGING (MAHALANOBIS + H-CLUSTER)             

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 2.5
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  output_dir            : results/04_tmp_gmm_component_merging_mt2p5

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,639
  original_components   : 26
  merged_components     : 12


In [6]:
# STEP4, OUR cohort assignment to merged GMM clusters (2x2 Composite)
import importlib
import scripts.our_assignment as our_assignment_mod

importlib.reload(our_assignment_mod)
from scripts.our_assignment import OURAssignmentConfig, run_our_assignment_to_merged_gmm

config_step4 = OURAssignmentConfig(
    output_dir="results/04_our_assignment",
    save_plot=True,
    save_tables=True,
    output_file="our_posterior_probabilities_merged.tsv",
    figure_file="our_assignment.png",
    case_label="CTEPH",
    control_label="AGP3K",
    bbj_alpha=0.20,
    verbose=True,
 )

step4_out = run_our_assignment_to_merged_gmm(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    label_map=label_map,
    merge_map=merge_map,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    training_use_zscale=config_step2.use_zscale,
    config=config_step4,
)

# Common downstream variables (kept for notebook convenience).
df_results = step4_out.df_results
probs_merged_our = step4_out.probs_merged_our
assigned_merged = step4_out.assigned_merged
assignment_conf = step4_out.assignment_confidence
cluster_stats = step4_out.cluster_stats


                STEP4: COHORT ASSIGNMENT TO MERGED GMM CLUSTERS                 

[CONFIGURATION]
--------------------------------------------------------------------------------
  output_dir            : results/04_our_assignment
  save_tables           : True
  save_plot             : True
  show_plot             : False

[RESULTS]
--------------------------------------------------------------------------------
  cohort rows           : 3,532
  merged_clusters (K)   : 7
  assignment_tsv        : results/04_our_assignment/our_posterior_probabilities_merged.tsv


In [7]:
# STEP4 QC, Assignment Confidence Distribution (Config + Runner)
import importlib
import scripts.our_assignment as our_assignment_mod

importlib.reload(our_assignment_mod)
from scripts.our_assignment import (
    OURAssignmentConfidenceQCConfig,
    run_step4_assignment_confidence_qc,
 )

config_step4_qc = OURAssignmentConfidenceQCConfig(
    output_dir=config_step4.output_dir,
    figure_file="assignment_confidence_distribution.png",
    thresholds=(0.80, 0.90, 0.95),
    save_plot=True,
    verbose=True,
 )

qc_out = run_step4_assignment_confidence_qc(
    df_results=df_results,
    step4_config=config_step4,
    qc_config=config_step4_qc,
 )


                    STEP4 QC: CONFIDENCE STATISTICS SUMMARY                     
count         3532
mean      0.997970
std       0.023129
min       0.539877
25%       0.999998
50%       1.000000
75%       1.000000
max       1.000000
--------------------------------------------------------------------------------
High Confidence Sample Counts:
  • >= 0.80 :  3518 samples (99.6%)
  • >= 0.90 :  3511 samples (99.4%)
  • >= 0.95 :  3506 samples (99.3%)
  • >= 0.99 :  3470 samples (98.2%)


In [8]:
# STEP5, High-Confidence Subset Visualization (Config + Runner)
import importlib
import scripts.high_confidence_visualization as high_conf_vis_mod

importlib.reload(high_conf_vis_mod)
from scripts.high_confidence_visualization import (
    HighConfidenceVizConfig,
    run_high_confidence_assignment_visualization,
 )

config_step5_highconf = HighConfidenceVizConfig(
    output_dir="results/05_high_confidence_visualization",
    threshold=0.95,
    save_tables=True,
    output_file="our_posterior_probabilities_conf_ge_{threshold_tag}.tsv",
    figure_file="our_assignment_conf_ge_{threshold_tag}.png",
    save_plot=True,
    verbose=True,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    bbj_alpha=config_step4.bbj_alpha,
 )

step5_out = run_high_confidence_assignment_visualization(
    bbj_samples_gmm=bbj_samples_gmm,
    df_results=df_results,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    merge_map=merge_map,
    eigenval=eigenval,
    step4_config=config_step4,
    config=config_step5_highconf,
 )

# Filtered STEP4 results (high-confidence only)
df_results_highconf = step5_out.df_results_highconf


                  STEP5: HIGH-CONFIDENCE SUBSET VISUALIZATION                   
  threshold             : 0.9500
  n_total               : 3,532
  n_kept                : 3,506
  kept_fraction         : 99.264%
  output_tsv            : results/05_high_confidence_visualization/our_posterior_probabilities_conf_ge_0p95.tsv


In [9]:
# Selection & Statistics
import importlib
import scripts.cluster_all_pcs_kde as cluster_all_pcs_kde_mod

importlib.reload(cluster_all_pcs_kde_mod)
from scripts.cluster_all_pcs_kde import ClusterAllPCsKDEConfig, run_cluster_all_pcs_kde

config_step6 = ClusterAllPCsKDEConfig(
    cluster_id=3,
    threshold=config_step5_highconf.threshold,
    output_dir="results/06_cluster3_all_pcs_kde_allpcs",
    save_tables=True,
    save_plot=True,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    verbose=True,
 )

step6_out = run_cluster_all_pcs_kde(
    df_results_highconf=df_results_highconf,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    eigenval=eigenval,
    step4_config=config_step4,
    config=config_step6,
 )

>>> ANALYZING ALL PC DISTRIBUTIONS FOR HIGH CONFIDENCE SAMPLES (CLUSTER 3)...
   -> confidence_threshold = 0.95
   -> output_dir = results/06_cluster3_all_pcs_kde_allpcs
   -> Cluster 3 samples: 3050 / 3506
   -> Case samples (Cluster 3): 397
   -> Control samples (Cluster 3): 2653
   -> Total PCs to analyze: 20
   -> Running statistical tests...
   -> Applying FDR correction (Benjamini-Hochberg method) to all tests...
   -> FDR correction complete.
      • Significant by t-test: 2 PC(s)
      • Significant by Mann-Whitney U: 3 PC(s)
                   ALL PC DISTRIBUTION STATISTICS - CLUSTER 3                   

PC1 (39.1%) Statistics:
  AGP3K (n=2653):
    Mean:   0.0060  |  Std:   0.0072
    Min:   -0.0167  |  Max:   0.0247
  CTEPH (n=397):
    Mean:   0.0065  |  Std:   0.0045
    Min:   -0.0075  |  Max:   0.0191
  Difference (CTEPH - AGP3K):   0.0005

PC2 (7.4%) Statistics:
  AGP3K (n=2653):
    Mean:   0.0028  |  Std:   0.0084
    Min:   -0.0308  |  Max:   0.0232
  CTEPH (n=397):